### 代码框 1：读取训练文本

这个代码框从 `input.txt` 中读取全部原始文本，并保存到变量 `text`。后面的建词表、编码、训练集划分和模型训练都基于这份文本。

In [ ]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

### 代码框 2：检查数据规模和样例

这个代码框打印文本总长度，并查看前 1000 个字符。作用是确认数据已经正确读入，同时对语料内容有一个直观感受。

In [ ]:
print(len(text))
print(text[:1000])

### 代码框 3：构建字符级词表

`set(text)` 提取语料中出现过的所有不同字符，`sorted(...)` 固定字符顺序，`vocab_size` 记录词表大小。这里采用的是字符级语言模型，所以每个字符就是一个 token。

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

### 代码框 4：建立字符和编号的双向映射

`map_char_to_id` 把字符映射成整数编号，`map_id_to_char` 把编号还原成字符。`encode` 负责把字符串变成 token id 序列，`decode` 负责把 token id 序列还原为字符串。模型只能处理数字，所以这是文本进入神经网络前的编码步骤。

In [ ]:
map_char_to_id = {ch: i for i, ch in enumerate(chars)}
map_id_to_char = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [map_char_to_id[ch] for ch in s]
decode = lambda l: ''.join([map_id_to_char[id] for id in l])
print(encode("hello world"))
print(decode(encode("hello world")))

### 代码框 5：把全文编码成 PyTorch 张量

这个代码框先导入 PyTorch，然后用 `encode(text)` 把整篇文本转换成整数序列，再转成 `torch.long` 类型的张量 `data`。后续的 embedding 层要求输入是整数 token id，因此这里必须使用 long 类型。

In [ ]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)

### 代码框 6：划分训练集和验证集

这个代码框把前 90% 的 token 作为 `train_data`，后 10% 作为 `val_data`。训练集用于更新模型参数，验证集用于之后评估模型在未见文本上的表现。

In [ ]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

### 代码框 7：理解上下文和预测目标

`block_size = 8` 表示一次最多用 8 个 token 作为上下文。`x` 是输入序列，`y` 是向右错开一位的目标序列。循环展示了语言模型的训练任务：给定前面的上下文 token，预测下一个 token。

In [ ]:
block_size = 8
train_data[:block_size + 1]
x = train_data[:block_size]
y = train_data[1:block_size + 1]
for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print(f"when input is{context} the target: {target}")

### 代码框 8：随机采样一个训练 batch

`get_batch` 会从训练集或验证集中随机选取多个起点，每个起点截取一段长度为 `block_size` 的输入 `x`，并取它右移一位后的序列作为目标 `y`。最终 `xb` 和 `yb` 的形状都是 `(batch_size, block_size)`，表示一次并行训练多段上下文。

In [ ]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(type):
    data = train_data if type == 'train' else val_data
    pos = torch.randint(len(data) - block_size, (batch_size, ))
    x = torch.stack([data[i:i + block_size] for i in pos])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in pos])
    return x, y

xb, yb = get_batch('train')
print("inputs:")
print(xb.shape)
print(xb)
print("targets:")
print(yb.shape)
print(yb)

### 代码框 9：定义 Bigram 语言模型并生成文本

这个代码框定义了一个最简单的字符级语言模型 `BigramLanguageModel`。`nn.Embedding(vocab_size, vocab_size)` 相当于一张查表矩阵：输入当前 token id，直接得到对下一个 token 的预测分数 `logits`。

`forward` 在训练时接收 `idx` 和 `targets`，先得到形状为 `(B, T, vocab_size)` 的 logits，再展平成 `(B*T, vocab_size)`，用交叉熵 `F.cross_entropy` 计算预测下一个 token 的损失。

`generate` 用自回归方式生成文本：把当前上下文送入模型，只取最后一个位置的 logits，用 softmax 转成概率分布，再用 `torch.multinomial` 采样下一个 token，并把它拼接回上下文。这个过程重复 `max_new_tokens` 次，最后用 `decode` 把生成的 token id 序列还原成文本。

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F


class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

m = BigramLanguageModel(vocab_size)
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

### 代码框 10：训练 Bigram 模型

这个代码框把 `batch_size` 调整为 32，然后循环训练 10000 步。每一步都会随机采样一个 batch，执行前向传播得到 `loss`，清空旧梯度，反向传播计算新梯度，再用优化器更新 embedding 表中的参数。每隔 1000 步打印一次当前 loss，用来观察训练是否在下降。

In [ ]:
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if steps % 1000 == 0:
        print(f"step {steps}: loss {loss.item()}")

### 代码框 11：用训练后的模型生成文本

这个代码框用全 0 token 作为起始上下文，调用 `m.generate(...)` 自回归生成 100 个新 token。生成结果仍然是 token id 序列，所以最后用 `decode` 把它转换回可读字符串。

In [27]:
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))





GRIONAMy the sth,
Busisellourre wece leer.
itiseh hallar, fooaind be:

'ly, agou?
OMAUCENThupee t


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
x.shape

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)
q = query(x)
wei = q @ k.transpose(-2, -1)

tril = torch.tril(torch.ones((T, T)))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
out = wei @ x

v = value(x)
out = wei @ v


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In [ ]:
class BatchNormld:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        xmean = x.mean(1, keepdim=True)
        xvar = x.var(1, keepdim=True)
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta
        return self.out
    
    def parameters(self):
        return [self.gamma, self.beta]

torch.manual_seed(1337)
module = BatchNormld(100)
x = torch.randn(32, 100)
x = module(x)
x.shape